## Vacuum table
Delete the data kept around for timetravel

In [ ]:
table_name = "bronze.wikipedia_page_reads"
partition_col = "date"

In [ ]:
from tfdslib.spark import get_spark_session
from delta.tables import DeltaTable

spark = get_spark_session('spark mgmt notebook')
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print('Vacuuming table')
delta_table = DeltaTable.forName(spark, table_name)
delta_table.vacuum(2)

print('All done.')

## Compact table
Reduce number of small files

In [ ]:
from pyspark.sql.functions import col
from tfdslib.spark import get_spark_session

spark = get_spark_session('spark mgmt notebook')
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

df = spark.table(table_name)
print('Determining partitions to process.')
distinct_partitions = df.select(partition_col).distinct().collect()

print(f'Iterating {len(distinct_partitions)} partitions.')
for row in distinct_partitions:
    dt_val = row[partition_col]
    print(f"Compacting partition: {dt_val}")

    df_part = df.filter(col(partition_col) == dt_val)

    df_part.coalesce(1).write \
        .mode("overwrite") \
        .option("dataChange", "false") \
        .insertInto(table_name, overwrite=True)
print('All done.')